<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Final_Project/MSE1003_Final_Project_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Comment out the below cells when running in Colab


In [ ]:
!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/

In [ ]:
import os
repo = "MSE1003H_RijaAnsari"
print(os.listdir(repo))


In [ ]:
%cd MSE1003H_RijaAnsari/Final_Project

# Final Project: Community Science and Self-Driving Labs

MSE1003 Goal: For a set of proposed 3D printing experiments from a community scientist, develop an
acquisition policy that maximizes (1) scientific discovery and (2) community engagement. There will be five
opportunities to iterate.

Scientific Discovery:

According to the Stanford Encyclopedia of Philosophy, "scientific discovery is the process or product of successful scientific inquiry. Objects of discovery can be things, events, processes, causes and properties as well as theories and hypotheses and their features."

For the purposes of this project, scientific discovery will be defined as experiments that explore novel areas in the feature space, have a high predicted target variable y and have high-model uncertainty. This will be quantified with large distance from current experiments, a strong energy absorption and most learning from the upper confidence bounded points.

Community engagemnent is defined as "seeks to better engage the community to achieve long-term and sustainable outcomes, processes, relationships, discourse, decision-making or implementation. To be successful, it must encompass strategies and processes that are senstive to the community context in which it occurs."

For this project, experiments that engage the community should be interpretable, integrate vast community interests, and have a simple and justifiable explanation. 

https://plato.stanford.edu/entries/scientific-discovery/

https://aese.psu.edu/research/centers/cecd/engagement-toolbox/engagement/what-is-community-engagement


## Iteration 1

In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor 
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel


from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from scipy.stats import norm


In [ ]:
#install botorch quietly
!pip install botorch -q

In [ ]:
# ─── Run this first to check your BoTorch version ─────────────────────────────
import botorch, gpytorch, torch
print(f"BoTorch:  {botorch.__version__}")
print(f"GPyTorch: {gpytorch.__version__}")
print(f"PyTorch:  {torch.__version__}")

In [ ]:
import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal

from gpytorch.models import ExactGP
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from gpytorch.distributions import MultitaskMultivariateNormal
from gpytorch.means import MultitaskMean, ConstantMean
from gpytorch.kernels import ScaleKernel, RBFKernel, MultitaskKernel

from botorch.models.model import Model
from botorch.posteriors.gpytorch import GPyTorchPosterior
from botorch.acquisition.objective import GenericMCObjective
from botorch.acquisition.monte_carlo import qNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf

Multi-Objective Optimization - Acquisition Scoring

S(x)=α⋅UCBnorm​(x)+β⋅dnorm​(x)

where alpha = 0.6 and beta = 0.4

Objective 1: Scientific Discovery

UCB(x)=μ(x)+κ⋅σ(x),κ=2.0

Objective 2: Community Engagement

d(x)=k1​i=1∑k​∥x−xi​∥2​,k=20

Weight Rationale


Both components are min-max normalised to [0,1][0, 1]
[0,1] before combining so neither objective dominates by scale. α=0.6\alpha = 0.6
α=0.6 prioritises model-driven discovery as the primary objective; β=0.4\beta = 0.4
β=0.4 ensures geographic spread across feature space, satisfying the community engagement requirement for diverse, interpretable experiment selection.


In [ ]:
#read data

data = pd.read_csv("Data.csv")

In [ ]:
data.head()

In [ ]:
training = data.copy()


In [ ]:
def objective_scientific_discovery(X_scaled, gp, kappa=2.0):
    """
    Objective 1: Scientific Discovery
    UCB(x) = mu(x) + kappa * sigma(x)
    
    Parameters
    ----------
    X_scaled : np.ndarray, shape (n_samples, n_features) — scaled feature matrix
    gp       : fitted GaussianProcessRegressor
    kappa    : float — exploration-exploitation trade-off (default 2.0)

    Returns
    -------
    ucb      : np.ndarray, shape (n_samples,) — raw UCB scores
    mu       : np.ndarray, shape (n_samples,) — posterior mean predictions
    sigma    : np.ndarray, shape (n_samples,) — posterior standard deviations
    """
    mu, sigma = gp.predict(X_scaled, return_std=True)
    ucb = mu + kappa * sigma
    return ucb, mu, sigma


def objective_community_engagement(X_scaled, n_components=5, k=20, random_state=1003):
    """
    Objective 2: Community Engagement
    d(x) = (1/k) * sum_{i=1}^{k} ||x - x_i||_2
    
    Mean distance to k nearest neighbours in PCA-reduced feature space.
    Large d(x) indicates a sparse, underexplored region of the geometry space.

    Parameters
    ----------
    X_scaled    : np.ndarray, shape (n_samples, n_features) — scaled feature matrix
    n_components: int   — number of PCA components (default 5)
    k           : int   — number of nearest neighbours (default 20)
    random_state: int   — random seed

    Returns
    -------
    d           : np.ndarray, shape (n_samples,) — mean k-NN distances
    X_pca       : np.ndarray, shape (n_samples, n_components) — PCA-reduced features
    """
    pca = PCA(n_components=n_components, random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)

    nn = NearestNeighbors(n_neighbors=k + 1)  # +1 to exclude self
    nn.fit(X_pca)
    distances, _ = nn.kneighbors(X_pca)
    d = distances[:, 1:].mean(axis=1)  # exclude self (distance = 0)
    return d, X_pca


def combined_acquisition_score(X_scaled, gp, alpha=0.6, beta=0.4, kappa=2.0, k=20, n_components=5):
    """
    Combined acquisition score:
    S(x) = alpha * UCB_norm(x) + beta * d_norm(x)

    Parameters
    ----------
    X_scaled    : np.ndarray — scaled feature matrix
    gp          : fitted GaussianProcessRegressor
    alpha       : float — weight for scientific discovery (default 0.6)
    beta        : float — weight for community engagement (default 0.4)
    kappa       : float — UCB exploration parameter (default 2.0)
    k           : int   — number of nearest neighbours for d(x) (default 20)
    n_components: int   — PCA components for d(x) (default 5)

    Returns
    -------
    score       : np.ndarray — combined acquisition scores
    mu          : np.ndarray — GP posterior means
    sigma       : np.ndarray — GP posterior standard deviations
    """
    assert np.isclose(alpha + beta, 1.0), "alpha + beta must sum to 1"

    ucb, mu, sigma = objective_scientific_discovery(X_scaled, gp, kappa=kappa)
    d, _ = objective_community_engagement(X_scaled, n_components=n_components, k=k)

    ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
    d_norm   = (d   - d.min())   / (d.max()   - d.min())

    score = alpha * ucb_norm + beta * d_norm
    return score, mu, sigma

In [ ]:
class SurrogateGP:
    """
    Gaussian Process surrogate model for multi-objective Bayesian optimisation.

    Parameters
    ----------
    n_train      : int   — number of points to subsample for fitting (default 2000)
    n_strata     : int   — number of quantile strata for stratified sampling (default 20)
    kappa        : float — UCB exploration-exploitation parameter (default 2.0)
    n_components : int   — PCA components for community engagement objective (default 5)
    k_neighbors  : int   — k for mean k-NN distance d(x) (default 20)
    alpha        : float — weight for scientific discovery in S(x) (default 0.6)
    beta         : float — weight for community engagement in S(x) (default 0.4)
    random_state : int   — random seed (default 1003)
    """

    def __init__(
        self,
        n_train=2000,
        n_strata=20,
        kappa=2.0,
        n_components=5,
        k_neighbors=20,
        alpha=0.6,
        beta=0.4,
        random_state=1003,
    ):
        assert np.isclose(alpha + beta, 1.0), "alpha + beta must sum to 1"

        self.n_train      = n_train
        self.n_strata     = n_strata
        self.kappa        = kappa
        self.n_components = n_components
        self.k_neighbors  = k_neighbors
        self.alpha        = alpha
        self.beta         = beta
        self.random_state = random_state

        self.scaler_ = StandardScaler()
        self.gp_     = GaussianProcessRegressor(
            kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
            n_restarts_optimizer=5,
            normalize_y=True,
            random_state=random_state,
        )
        self.is_fitted_ = False

    def _stratified_subsample(self, y):
        np.random.seed(self.random_state)
        bins = np.percentile(y, np.linspace(0, 100, self.n_strata + 1))
        indices = []
        per_stratum = max(1, self.n_train // self.n_strata)
        for i in range(self.n_strata):
            mask = (y >= bins[i]) & (y < bins[i + 1])
            candidates = np.where(mask)[0]
            chosen = np.random.choice(
                candidates, min(per_stratum, len(candidates)), replace=False
            )
            indices.extend(chosen)
        return np.array(indices)


    #def fit(self, X, y):
     #   X_scaled = self.scaler_.fit_transform(X)
      #  idx_train = self._stratified_subsample(y)

#        self.gp_.fit(X_scaled[idx_train], y[idx_train])
 #       self.X_scaled_ = X_scaled
  #      self.y_        = y
   #     self.is_fitted_ = True
#
 #       print(f"GP fitted on {len(idx_train)} / {len(y)} points.")
  #      print(f"Optimised kernel: {self.gp_.kernel_}")
   #     return self

    def fit(self, X, y):
        self.gp_.fit(X, y)
        self.X_scaled_ = X
        self.y_        = y
        self.is_fitted_ = True

        print(f"GP fitted on {len(y)} / {len(y)} points.")
        print(f"Optimised kernel: {self.gp_.kernel_}")
        return self

    def score(self):
        if not self.is_fitted_:
            raise RuntimeError("Call .fit(X, y) before scoring.")

        ucb, mu, sigma = objective_scientific_discovery(self.X_scaled_, self.gp_, kappa=self.kappa)
        d, _           = objective_community_engagement(self.X_scaled_, n_components=self.n_components, k=self.k_neighbors)
        S, mu, sigma   = combined_acquisition_score(self.X_scaled_, self.gp_,
                                                    alpha=self.alpha, beta=self.beta,
                                                    kappa=self.kappa, k=self.k_neighbors,
                                                    n_components=self.n_components)
        return S, mu, sigma

    def select_top_n(self, n=26):
        S, mu, sigma = self.score()
        top_idx = np.argsort(S)[::-1][:n]
        return top_idx, S, mu, sigma

In [ ]:
"""training = training.rename(columns={"Unnamed: 0": "Pool_index"})
features = ["x1","x2","x3","x4","x5","x6","x7","x8","x9","x10","x11"]

# ── Separate features and target ───────────────────────────────────────
X = training[features].values   # shape (13224, 11)
y = training["y"].values              # shape (13224,)

# ── Min-max scale X and y independently ────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)             # shape (13224, 11), range [0, 1]
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()  # shape (13224,), range [0, 1]

print(f"X shape : {X_scaled.shape}")
print(f"y shape : {y_scaled.shape}")
print(f"X range : [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"y range : [{y_scaled.min():.4f}, {y_scaled.max():.4f}]")"""

In [ ]:
from sklearn.model_selection import train_test_split

training = training.rename(columns={"Unnamed: 0": "Pool_index"})
features = ["x1","x2","x3","x4","x5","x6","x7","x8","x9","x10","x11"]

X = training[features].values
y = training["y"].values

# ── Scalers fitted on full data so pool scoring stays consistent ───────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

# ── Stratified subsample for GP fitting (keeps y distribution intact) ──
N_TRAIN = 3000   # increase to 3000-4000 if accuracy is insufficient

np.random.seed(1003)

# Shuffle the full dataset first, then stratify
shuffle_idx = np.random.permutation(len(y_scaled))
X_shuffled  = X_scaled[shuffle_idx]
y_shuffled  = y_scaled[shuffle_idx]

# Stratified subsample on shuffled data
bins      = np.percentile(y_shuffled, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_shuffled >= bins[i]) & (y_shuffled < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)

train_idx = np.array(train_idx)

X_train = X_shuffled[train_idx]
y_train = y_shuffled[train_idx]

# Keep full shuffled pool for scoring, with matching Pool_index
training_shuffled = training.iloc[shuffle_idx].reset_index(drop=True)

print(f"GP training subset : {X_train.shape[0]} points")
print(f"Pool_index range in training subset: {training_shuffled.iloc[train_idx]['Pool_index'].min()} – {training_shuffled.iloc[train_idx]['Pool_index'].max()}")


"""bins      = np.percentile(y_scaled, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_scaled >= bins[i]) & (y_scaled < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)
train_idx = np.array(train_idx)

X_train = X_scaled[train_idx]
y_train = y_scaled[train_idx]
"""
print(f"Full pool : {X_scaled.shape[0]} points (used for scoring)")
print(f"GP training subset : {X_train.shape[0]} points (used for fitting)")
print(f"X range : [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"y range : [{y_scaled.min():.4f}, {y_scaled.max():.4f}]")

In [ ]:
model = SurrogateGP(random_state=1003)
model.fit(X_train, y_train)

model.X_scaled_ = X_shuffled
model.y_        = y_shuffled

top_idx, S, mu, sigma = model.select_top_n(n=26)

In [ ]:
top_idx

In [ ]:
def make_explanation(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    return (
        f"Selected for high combined acquisition score (S={row['score']:.3f}) "
        f"driven by predicted energy absorption mu={row['mu']:.3f} and model uncertainty sigma={row['sigma']:.3f} "
        f"(scientific discovery), and high feature-space diversity (community engagement). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

# ── Build submission dataframe ─────────────────────────────────────────
submission = training.iloc[top_idx].copy().reset_index(drop=True)
submission = submission.rename(columns={name: f"x{i+1}" for i, name in enumerate(features)})

submission["mu"]    = mu[top_idx]
submission["sigma"] = sigma[top_idx]
submission["score"] = S[top_idx]
submission["explanation"] = submission.apply(make_explanation, axis=1)
submission["sis_id"] = "ansarir5"

# ── Reorder to match submission spec ──────────────────────────────────
feat_cols  = [f"x{i}" for i in range(1, 12)]
submission = submission[["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]]

submission = submission.sort_values("score", ascending=False).reset_index(drop=True)

print(submission[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

In [ ]:
submission

In [ ]:
submission.to_csv("iteration1_submission2.csv", index=False)
print("Saved.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Gather scores and objectives for all points ────────────────────────
S, mu, sigma = model.score()
d, X_pca     = d, X_pca = objective_community_engagement(model.X_scaled_, n_components=model.n_components, k=model.k_neighbors)
#model.objective_community_engagement()

ucb          = mu + model.kappa * sigma
ucb_norm     = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm       = (d   - d.min())   / (d.max()   - d.min())

# ── Pareto frontier (maximise both objectives) ─────────────────────────
# A point is non-dominated if no other point is better in both objectives
objectives   = np.stack([ucb_norm, d_norm], axis=1)   # (13224, 2)

def is_non_dominated(obj):
    """Returns boolean mask of non-dominated points (maximisation)."""
    n        = len(obj)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(obj[j] >= obj[i]) and np.any(obj[j] > obj[i]):
                dominated[i] = True
                break
    return ~dominated

is_pareto    = is_non_dominated(objectives[np.argsort(S)[::-1][:500]])  # check on top 500 for speed
pareto_idx   = np.argsort(S)[::-1][:500][is_pareto]
top26_mask   = np.isin(np.arange(len(S)), top_idx)

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 1 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])   # full-width: score vs index
ax_o = fig.add_subplot(gs[1, 0])   # objective space: UCB vs diversity
ax_p = fig.add_subplot(gs[1, 1])   # PCA coloured by S

# ─── Panel 1: Acquisition score across all pool points ────────────────
ax_s.scatter(np.arange(len(S)), S,
             c=S, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top_idx, S[top_idx],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")
ax_s.scatter(pareto_idx, S[pareto_idx],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score Across Full Pool")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
# All points (faded)
ax_o.scatter(ucb_norm, d_norm,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")

# Pareto-non-dominated (top 500 subset)
ax_o.scatter(ucb_norm[pareto_idx], d_norm[pareto_idx],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")

# Top 26 selected
ax_o.scatter(ucb_norm[top_idx], d_norm[top_idx],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")

# Pareto staircase
pareto_pts = np.stack([ucb_norm[pareto_idx], d_norm[pareto_idx]], axis=1)
sorted_p   = pareto_pts[np.argsort(pareto_pts[:, 0])]
ax_o.step(sorted_p[:, 0], sorted_p[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")

ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space coloured by acquisition score ─────────────────
sc = ax_p.scatter(X_pca[:, 0], X_pca[:, 1],
                  c=S, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca[pareto_idx, 0], X_pca[pareto_idx, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca[top_idx, 0], X_pca[top_idx, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")

#plt.savefig("/mnt/user-data/outputs/iteration1_pareto.png", dpi=150, bbox_inches="tight")
#plt.show()
#print("Saved.")

There is a heavy selection on the pareto non-dominated samples for the first 26 samples. 

For the next selection, the acquisition policy needs to be modified to emphasize community engagement. 

In [ ]:
"""from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ── Train/test split for predictive accuracy ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=1003
)

# Refit a temporary GP on train split only for evaluation
gp_eval = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval.fit(X_train, y_train)
mu_test, sigma_test = gp_eval.predict(X_test, return_std=True)

rmse      = np.sqrt(mean_squared_error(y_test, mu_test))
r2        = 1 - np.sum((y_test - mu_test)**2) / np.sum((y_test - y_test.mean())**2)
mean_std  = sigma_test.mean()

print(f"Test RMSE : {rmse:.4f}")
print(f"Test R²   : {r2:.4f}")
print(f"Mean σ    : {mean_std:.4f}")

# ── Full pool predictions from fitted surrogate ────────────────────────
S, mu, sigma = model.score()
d, X_pca     = objective_community_engagement(
    model.X_scaled_, n_components=model.n_components, k=model.k_neighbors
)
ucb      = mu + model.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    "Iteration 1 — Surrogate Performance, Uncertainty & Acquisition Policy",
    fontsize=14, fontweight="bold", y=0.99
)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit  = fig.add_subplot(gs[0, :2])   # predicted vs actual
ax_res  = fig.add_subplot(gs[0, 2:])   # residuals
ax_ucb  = fig.add_subplot(gs[1, 0])    # UCB distribution
ax_std  = fig.add_subplot(gs[1, 1])    # uncertainty distribution
ax_div  = fig.add_subplot(gs[1, 2])    # diversity distribution
ax_s    = fig.add_subplot(gs[1, 3])    # combined score distribution

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_test, mu_test, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_test.min(), mu_test.min()), max(y_test.max(), mu_test.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse:.4f}  R²={r2:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test.min(), sigma_test.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals = y_test - mu_test
ax_res.scatter(mu_test, residuals, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test.min(), mu_test.max(), 100),
    -2 * sigma_test.mean(), 2 * sigma_test.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper: distribution panel ───────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 26 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6: Score component distributions ────────────────────────
dist_panel(ax_ucb, ucb_norm,  ucb_norm[top_idx],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution", "#3498db")
dist_panel(ax_std, sigma,     sigma[top_idx],
           "σ(x)",            "Model Uncertainty\nσ distribution",      "#9b59b6")
dist_panel(ax_div, d_norm,    d_norm[top_idx],
           "$d_{norm}$(x)",   "Community Engagement\nDiversity distribution", "#2ecc71")
dist_panel(ax_s,   S,         S[top_idx],
           "S(x)",            "Combined Acquisition\nScore S(x)",        "#e67e22")

plt.savefig("/mnt/user-data/outputs/iteration1_performance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")"""

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ── Train/test split on augmented subsampled data ──────────────────────
X_tr_eval, X_te_eval, y_tr_eval, y_te_eval = train_test_split(
    X_train, y_train, test_size=0.2, random_state=1003
)

gp_eval = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval.fit(X_tr_eval, y_tr_eval)
mu_test, sigma_test = gp_eval.predict(X_te_eval, return_std=True)

rmse     = np.sqrt(mean_squared_error(y_te_eval, mu_test))
r2       = 1 - np.sum((y_te_eval - mu_test)**2) / np.sum((y_te_eval - y_te_eval.mean())**2)
mean_std = sigma_test.mean()

print(f"Test RMSE : {rmse:.4f}")
print(f"Test R²   : {r2:.4f}")
print(f"Mean σ    : {mean_std:.4f}")

# ── Full augmented pool predictions from fitted surrogate ──────────────
S, mu, sigma = model_iter2.score()
d, X_pca     = objective_community_engagement(
    model_iter2.X_scaled_, n_components=model_iter2.n_components, k=model_iter2.k_neighbors
)
ucb      = mu + model_iter2.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    "Iteration 2 — Surrogate Performance, Uncertainty & Acquisition Policy",
    fontsize=14, fontweight="bold", y=0.99
)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval, mu_test, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval.min(), mu_test.min()), max(y_te_eval.max(), mu_test.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse:.4f}  R²={r2:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test.min(), sigma_test.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals = y_te_eval - mu_test
ax_res.scatter(mu_test, residuals, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test.min(), mu_test.max(), 100),
    -2 * sigma_test.mean(), 2 * sigma_test.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper: distribution panel ───────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 20 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6: Score component distributions ────────────────────────
dist_panel(ax_ucb, ucb_norm,  ucb_norm[top_idx],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",      "#3498db")
dist_panel(ax_std, sigma,     sigma[top_idx],
           "σ(x)",            "Model Uncertainty\nσ distribution",            "#9b59b6")
dist_panel(ax_div, d_norm,    d_norm[top_idx],
           "$d_{norm}$(x)",   "Community Engagement\nDiversity distribution", "#2ecc71")
dist_panel(ax_s,   S,         S[top_idx],
           "S(x)",            "Combined Acquisition\nScore S(x)",             "#e67e22")

#plt.savefig("/mnt/user-data/outputs/iteration2_performance.png", dpi=150, bbox_inches="tight")
#plt.show()
#print("Saved.")

## Acquisition Policy

Now that we have received 13 proposals, we have to select the top 20 points for evaluation. 

In [ ]:
# ── Diagnostic: check column names across all files ───────────────────
for fpath in proposal_files:
    tmp = pd.read_csv(fpath)
    print(f"{os.path.basename(fpath)}: {tmp.columns.tolist()}")

In [ ]:
"""import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ── 1. Load all 13 proposal CSVs ───────────────────────────────────────
PROPOSAL_DIR = "Proposals_iter1/"

proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "proposals_*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

# ── 2. Normalise column names and pool all proposals ───────────────────
COL_MAP = {
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled proposals : {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.groupby("scientist_id").size().to_string())

# ── 3. Join back to original dataset to recover x1–x11 and true y ─────
proposals_full = proposals_pooled.merge(
    training[["Pool_index"] + features + ["y"]],
    on="Pool_index",
    how="left"
)

n_missing = proposals_full[features[0]].isnull().sum()
print(f"\nUnmatched Pool_index entries : {n_missing}")
print(f"proposals_full shape         : {proposals_full.shape}")

# ── 4. Clean inf values in community scores ────────────────────────────
proposals_full["community_score"] = proposals_full["community_score"].replace(
    [np.inf, -np.inf], np.nan
)
proposals_full["community_score"] = proposals_full.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))

print(f"Inf remaining  : {np.isinf(proposals_full['community_score']).sum()}")
print(f"NaN remaining  : {proposals_full['community_score'].isnull().sum()}")

# ── 5. Deduplicate — one row per Pool_index ────────────────────────────
proposals_deduped = (
    proposals_full
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"\nUnique proposals after dedup : {len(proposals_deduped)}")

# ── 6. Scale proposals ─────────────────────────────────────────────────
X_proposals_scaled = scaler_X.transform(proposals_deduped[features].values)
y_proposals_scaled = scaler_y.transform(
    proposals_deduped["y"].values.reshape(-1, 1)
).ravel()

# ── 7. Build augmented pool (used for scoring) ─────────────────────────
X_pool_augmented  = np.vstack([X_shuffled, X_proposals_scaled])
y_pool_augmented  = np.concatenate([y_shuffled, y_proposals_scaled])
pool_index_augmented = np.concatenate([
    training_shuffled["Pool_index"].values,
    proposals_deduped["Pool_index"].values
])

print(f"\nAugmented pool : {X_pool_augmented.shape[0]} points")
print(f"  Original     : {X_shuffled.shape[0]}")
print(f"  Proposals    : {X_proposals_scaled.shape[0]}")

# ── 8. Shuffle augmented pool ──────────────────────────────────────────
np.random.seed(1003)
shuffle_idx      = np.random.permutation(len(y_pool_augmented))
X_aug_shuf       = X_pool_augmented[shuffle_idx]
y_aug_shuf       = y_pool_augmented[shuffle_idx]
pool_idx_shuf    = pool_index_augmented[shuffle_idx]

# ── 9. Stratified subsample for GP fitting ─────────────────────────────
bins      = np.percentile(y_aug_shuf, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_aug_shuf >= bins[i]) & (y_aug_shuf < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)
train_idx = np.array(train_idx)

# ── 10. Add proposal points not already in training subset ────────────
train_pool_indices = set(pool_idx_shuf[train_idx])
new_proposal_mask  = ~proposals_deduped["Pool_index"].isin(train_pool_indices)
new_proposals      = proposals_deduped[new_proposal_mask]

X_new = scaler_X.transform(new_proposals[features].values)
y_new = scaler_y.transform(new_proposals["y"].values.reshape(-1, 1)).ravel()

X_train_aug = np.vstack([X_aug_shuf[train_idx], X_new])
y_train_aug = np.concatenate([y_aug_shuf[train_idx], y_new])

print(f"\nRandom subset  : {len(train_idx)} points")
print(f"New proposals  : {X_new.shape[0]} points")
print(f"Final training : {X_train_aug.shape[0]} points")
print(f"Pool_index range in training: "
      f"{pool_idx_shuf[train_idx].min()} – {pool_idx_shuf[train_idx].max()}")

# ── 11. Refit surrogate ────────────────────────────────────────────────
model_iter2 = SurrogateGP(random_state=1003)
model_iter2.fit(X_train_aug, y_train_aug)
model_iter2.X_scaled_ = X_aug_shuf
model_iter2.y_        = y_aug_shuf"""

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ── 1. Load all 13 proposal CSVs ───────────────────────────────────────
PROPOSAL_DIR = "Proposals_iter1/"

proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "proposals_*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

# ── 2. Normalise column names and pool all proposals ───────────────────
COL_MAP = {
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled proposals : {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.groupby("scientist_id").size().to_string())

# ── 3. Join back to original dataset to recover x1–x11 and true y ─────
proposals_full = proposals_pooled.merge(
    training[["Pool_index"] + features + ["y"]],
    on="Pool_index",
    how="left"
)

n_missing = proposals_full[features[0]].isnull().sum()
print(f"\nUnmatched Pool_index entries : {n_missing}")
print(f"proposals_full shape         : {proposals_full.shape}")

# ── 4. Clean inf and nan values in community scores ────────────────────
proposals_full["community_score"] = proposals_full["community_score"].replace(
    [np.inf, -np.inf], np.nan
)

# Per-scientist median fill for scientists with some finite values
proposals_full["community_score"] = proposals_full.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))

# Global fallback for scientists where ALL scores were inf/nan
global_median = proposals_full["community_score"].median()
proposals_full["community_score"] = proposals_full["community_score"].fillna(global_median)

print(f"Inf remaining  : {np.isinf(proposals_full['community_score']).sum()}")
print(f"NaN remaining  : {proposals_full['community_score'].isnull().sum()}")
print(f"Global median fallback : {global_median:.4f}")

# ── 5. Deduplicate — one row per Pool_index ────────────────────────────
proposals_deduped = (
    proposals_full
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"\nUnique proposals after dedup : {len(proposals_deduped)}")

# ── 6. Scale proposals ─────────────────────────────────────────────────
X_proposals_scaled = scaler_X.transform(proposals_deduped[features].values)
y_proposals_scaled = scaler_y.transform(
    proposals_deduped["y"].values.reshape(-1, 1)
).ravel()

# ── 7. Build augmented pool (used for scoring) ─────────────────────────
X_pool_augmented     = np.vstack([X_shuffled, X_proposals_scaled])
y_pool_augmented     = np.concatenate([y_shuffled, y_proposals_scaled])
pool_index_augmented = np.concatenate([
    training_shuffled["Pool_index"].values,
    proposals_deduped["Pool_index"].values
])

print(f"\nAugmented pool : {X_pool_augmented.shape[0]} points")
print(f"  Original     : {X_shuffled.shape[0]}")
print(f"  Proposals    : {X_proposals_scaled.shape[0]}")

# ── 8. Shuffle augmented pool ──────────────────────────────────────────
np.random.seed(1003)
shuffle_idx          = np.random.permutation(len(y_pool_augmented))
X_aug_shuf           = X_pool_augmented[shuffle_idx]
y_aug_shuf           = y_pool_augmented[shuffle_idx]
pool_idx_shuf        = pool_index_augmented[shuffle_idx]

# ── 9. Stratified subsample for GP fitting ─────────────────────────────
bins      = np.percentile(y_aug_shuf, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_aug_shuf >= bins[i]) & (y_aug_shuf < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)
train_idx = np.array(train_idx)

# ── 10. Add proposal points not already in training subset ─────────────
train_pool_indices = set(pool_idx_shuf[train_idx])
new_proposal_mask  = ~proposals_deduped["Pool_index"].isin(train_pool_indices)
new_proposals      = proposals_deduped[new_proposal_mask]

X_new = scaler_X.transform(new_proposals[features].values)
y_new = scaler_y.transform(new_proposals["y"].values.reshape(-1, 1)).ravel()

X_train_aug = np.vstack([X_aug_shuf[train_idx], X_new])
y_train_aug = np.concatenate([y_aug_shuf[train_idx], y_new])

print(f"\nRandom subset  : {len(train_idx)} points")
print(f"New proposals  : {X_new.shape[0]} points")
print(f"Final training : {X_train_aug.shape[0]} points")

# ── 11. Refit surrogate ────────────────────────────────────────────────
model_iter2 = SurrogateGP(random_state=1003)
model_iter2.fit(X_train_aug, y_train_aug)
model_iter2.X_scaled_ = X_aug_shuf
model_iter2.y_        = y_aug_shuf

In [ ]:
# ── Diagnostic ────────────────────────────────────────────────────────
print(proposals_full.groupby("scientist_id")["community_score"]
      .apply(lambda x: x.replace([np.inf, -np.inf], np.nan).isnull().all())
      .to_string())

In [ ]:
"""import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ── 1. Load all 13 proposal CSVs ───────────────────────────────────────
PROPOSAL_DIR = "Proposals_iter1/"

proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "proposals_*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

# ── Column name map — normalise all variants to a standard schema ──────
COL_MAP = {
    # variant A
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    # variant B
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    # keep only the standardised columns
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"Pooled proposals: {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.groupby("scientist_id").size().to_string())

""""""# ── 2. Pool all proposals with community scientist labels ──────────────
proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp["scientist_id"] = scientist_id
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.rename(columns={"Index": "Pool_index"})
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled proposals: {proposals_pooled.shape[0]} rows from {proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.head(6).to_string(index=False))"""
"""
# ── 3. Join back to original dataset to recover x1–x11 and true y ─────
proposals_full = proposals_pooled.merge(
    training[["Pool_index"] + features + ["y"]],
    on="Pool_index",
    how="left"
)
#proposals_full = proposals_full.rename(columns={"Score": "community_score"})

n_missing = proposals_full[features[0]].isnull().sum()
print(f"\nUnmatched Pool_index entries: {n_missing}")
print(f"proposals_full shape: {proposals_full.shape}")
print(proposals_full[["Pool_index", "scientist_id", "community_score", "y"]].head(8).to_string(index=False))
"""
"""# ── 4. Scale proposals ─────────────────────────────────────────────────
X_proposals = proposals_full[features].values
y_proposals = proposals_full["y"].values

X_proposals_scaled = scaler_X.transform(X_proposals)
y_proposals_scaled = scaler_y.transform(y_proposals.reshape(-1, 1)).ravel()

# ── 5. Augment full pool (used for scoring) ────────────────────────────
X_pool_augmented = np.vstack([X_shuffled, X_proposals_scaled])
y_pool_augmented = np.concatenate([y_shuffled, y_proposals_scaled])

# Track Pool_index alignment for the augmented pool
pool_index_augmented = np.concatenate([
    training_shuffled["Pool_index"].values,
    proposals_full["Pool_index"].values
])

print(f"\nAugmented pool : {X_pool_augmented.shape[0]} points (used for scoring)")
print(f"  Original pool : {X_shuffled.shape[0]}")
print(f"  Proposals     : {X_proposals_scaled.shape[0]}")

# ── 6. Stratified subsample for GP fitting ─────────────────────────────
np.random.seed(1003)

shuffle_idx  = np.random.permutation(len(y_pool_augmented))
X_aug_shuf   = X_pool_augmented[shuffle_idx]
y_aug_shuf   = y_pool_augmented[shuffle_idx]
pool_idx_shuf = pool_index_augmented[shuffle_idx]

bins      = np.percentile(y_aug_shuf, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_aug_shuf >= bins[i]) & (y_aug_shuf < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)

train_idx = np.array(train_idx)
X_train   = X_aug_shuf[train_idx]
y_train   = y_aug_shuf[train_idx]

print(f"GP training subset : {X_train.shape[0]} points (shuffled, stratified)")
print(f"Pool_index range   : {pool_idx_shuf[train_idx].min()} – {pool_idx_shuf[train_idx].max()}")

# ── 7. Refit surrogate on augmented + subsampled data ─────────────────
model_iter2 = SurrogateGP(random_state=1003)
model_iter2.fit(X_train, y_train)

# Point surrogate at full augmented pool for scoring
model_iter2.X_scaled_ = X_aug_shuf
model_iter2.y_        = y_aug_shuf""""""

# Pool_index values already in the random training subset
train_pool_indices = set(pool_idx_shuf[train_idx])

# Proposal points not already in the training subset
new_proposal_mask = ~proposals_full["Pool_index"].isin(train_pool_indices)
new_proposals     = proposals_full[new_proposal_mask]

X_new = scaler_X.transform(new_proposals[features].values)
y_new = scaler_y.transform(new_proposals["y"].values.reshape(-1, 1)).ravel()

# Augment training set with new proposal points
X_train_aug = np.vstack([X_train, X_new])
y_train_aug = np.concatenate([y_train, y_new])

print(f"Random subset    : {X_train.shape[0]} points")
print(f"New proposals    : {X_new.shape[0]} points (not already in training)")
print(f"Final training   : {X_train_aug.shape[0]} points")

model_iter2 = SurrogateGP(random_state=1003)
model_iter2.fit(X_train_aug, y_train_aug)
model_iter2.X_scaled_ = X_aug_shuf
model_iter2.y_        = y_aug_shuf"""

In [ ]:
# ── Diagnostic: check community scores per scientist ──────────────────
print(proposals_full.groupby("scientist_id")["community_score"].describe().to_string())
print("\nInf counts per scientist:")
print(proposals_full.groupby("scientist_id")["community_score"]
      .apply(lambda x: np.isinf(x).sum()).to_string())

In [ ]:
def acquisition_score_iter2(
    model,
    proposals_full,
    pool_idx_shuf,
    alpha=0.4,    # scientific discovery (down from 0.6)
    beta=0.3,     # feature-space diversity
    gamma=0.2,    # community scientist score
    delta=0.1,    # scientist coverage
    kappa=2.0,
    n_components=5,
    k=20,
):
    """
    Updated acquisition score for iteration 2.

    S(x) = alpha * UCB_norm(x)
         + beta  * d_norm(x)
         + gamma * community_score_norm(x)
         + delta * scientist_coverage_norm(x)

    Only scores points that appear in proposals_full (the 200-point proposal pool).
    
    Parameters
    ----------
    model             : fitted SurrogateGP (scored over full augmented pool)
    proposals_full    : pd.DataFrame — pooled proposals with Pool_index, 
                        community_score, scientist_id, and x1–x11
    pool_idx_shuf     : np.ndarray — Pool_index values aligned to model.X_scaled_
    alpha             : weight for UCB (scientific discovery)
    beta              : weight for feature-space diversity
    gamma             : weight for community scientist score
    delta             : weight for scientist coverage
    """
    assert np.isclose(alpha + beta + gamma + delta, 1.0), "weights must sum to 1"

    # ── Locate proposal rows in the augmented pool ─────────────────────
    proposal_pool_indices = proposals_full["Pool_index"].values
    mask = np.isin(pool_idx_shuf, proposal_pool_indices)
    proposal_rows = np.where(mask)[0]   # positional indices into model.X_scaled_

    # Align proposals_full to the order they appear in model.X_scaled_
    idx_map = {pid: pos for pos, pid in zip(proposal_rows,
               pool_idx_shuf[proposal_rows])}
    aligned_pos = np.array([idx_map[pid] for pid in proposal_pool_indices
                            if pid in idx_map])
    aligned_proposals = proposals_full[
        proposals_full["Pool_index"].isin(pool_idx_shuf[aligned_pos])
    ].copy().reset_index(drop=True)

    X_prop = model.X_scaled_[aligned_pos]   # shape (n_proposals, 11)

    # ── Objective 1: Scientific Discovery — UCB ────────────────────────
    ucb, mu, sigma = objective_scientific_discovery(X_prop, model.gp_, kappa=kappa)
    ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min() + 1e-8)

    # ── Objective 2: Feature-space diversity — k-NN in PCA space ──────
    d, _ = objective_community_engagement(
        X_prop, n_components=n_components, k=min(k, len(X_prop) - 1)
    )
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)

    # ── Objective 3: Community scientist score ─────────────────────────
    """cs = aligned_proposals["community_score"].values.astype(float)
    cs_norm = (cs - cs.min()) / (cs.max() - cs.min() + 1e-8)
"""
    # ── Objective 3: Community scientist score ─────────────────────────────
    cs = aligned_proposals["community_score"].values.astype(float)

    # Replace inf with nan, then fill with column max of finite values
    cs = np.where(np.isinf(cs), np.nan, cs)
    finite_max = np.nanmax(cs)
    finite_min = np.nanmin(cs)
    cs = np.where(np.isnan(cs), finite_max, cs)   # treat inf as highest finite score

    cs_norm = (cs - finite_min) / (finite_max - finite_min + 1e-8)

    # ── Objective 4: Scientist coverage ───────────────────────────────
    # Reward points from scientists who are underrepresented so far.
    # Coverage score = 1 / (frequency of that scientist_id in pool)
    # so rare scientists get a higher score.
    scientist_ids   = aligned_proposals["scientist_id"].values
    id_counts       = pd.Series(scientist_ids).value_counts()
    coverage        = np.array([1.0 / id_counts[sid] for sid in scientist_ids])
    coverage_norm   = (coverage - coverage.min()) / (coverage.max() - coverage.min() + 1e-8)

    # ── Combined score ─────────────────────────────────────────────────
    S = (alpha * ucb_norm
       + beta  * d_norm
       + gamma * cs_norm
       + delta * coverage_norm)

    return S, mu, sigma, ucb_norm, d_norm, cs_norm, coverage_norm, aligned_proposals, aligned_pos

In [ ]:
# ── Fix 1: clean inf values in proposals_full before scoring ──────────
proposals_full["community_score"] = proposals_full["community_score"].replace(
    [np.inf, -np.inf], np.nan
)
# fill nan with the median score per scientist so each scientist's
# scale is preserved rather than pulling everything to a global value
proposals_full["community_score"] = proposals_full.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))

print("Inf values remaining:", np.isinf(proposals_full["community_score"]).sum())
print("NaN values remaining:", proposals_full["community_score"].isnull().sum())

# ── Fix 2: deduplicate — keep one row per Pool_index ──────────────────
# Aggregate duplicate Pool_index entries across scientists:
# - average the community scores
# - concatenate scientist_ids so coverage is tracked
proposals_deduped = (
    proposals_full
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"\nProposals after dedup: {len(proposals_deduped)} unique Pool_index values")
print(f"Original pooled rows : {len(proposals_full)}")

In [ ]:
(S_prop, mu_prop, sigma_prop,
 ucb_norm_prop, d_norm_prop,
 cs_norm_prop, cov_norm_prop,
 aligned_proposals, aligned_pos) = acquisition_score_iter2(
    model_iter2,
    proposals_deduped,
    pool_idx_shuf,
)

top20_local  = np.argsort(S_prop)[::-1][:20]   # indices into aligned_proposals
top20_pos    = aligned_pos[top20_local]          # positional indices into model.X_scaled_
top20_pool   = pool_idx_shuf[top20_pos]          # original Pool_index values

print(f"Top 20 Pool_index values: {top20_pool}")
print(aligned_proposals.iloc[top20_local][
    ["Pool_index", "scientist_id", "community_score"]
].to_string(index=False))

In [ ]:
S, mu, sigma = model_iter2.score()
d, X_pca     = objective_community_engagement(
    model_iter2.X_scaled_, n_components=model_iter2.n_components, k=model_iter2.k_neighbors
)
ucb      = mu + model_iter2.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

# Pareto on top 500 by score
objectives = np.stack([ucb_norm, d_norm], axis=1)
top500_idx = np.argsort(S)[::-1][:500]
is_pareto  = is_non_dominated(objectives[top500_idx])
pareto_idx = top500_idx[is_pareto]

# top20 positional indices into model_iter2.X_scaled_
top20_pos_plot = aligned_pos[top20_local]

fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 2 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])
ax_o = fig.add_subplot(gs[1, 0])
ax_p = fig.add_subplot(gs[1, 1])

# ─── Panel 1: Acquisition score across augmented pool ─────────────────
ax_s.scatter(np.arange(len(S)), S,
             c=S, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top20_pos_plot, S[top20_pos_plot],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_s.scatter(pareto_idx, S[pareto_idx],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score Across Augmented Pool")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
ax_o.scatter(ucb_norm, d_norm,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")
ax_o.scatter(ucb_norm[pareto_idx], d_norm[pareto_idx],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")
ax_o.scatter(ucb_norm[top20_pos_plot], d_norm[top20_pos_plot],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
pareto_pts = np.stack([ucb_norm[pareto_idx], d_norm[pareto_idx]], axis=1)
sorted_p   = pareto_pts[np.argsort(pareto_pts[:, 0])]
ax_o.step(sorted_p[:, 0], sorted_p[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")
ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space ───────────────────────────────────────────────
sc = ax_p.scatter(X_pca[:, 0], X_pca[:, 1],
                  c=S, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca[pareto_idx, 0], X_pca[pareto_idx, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca[top20_pos_plot, 0], X_pca[top20_pos_plot, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")

#plt.savefig("/mnt/user-data/outputs/iteration2_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
#print("Saved.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ── Train/test split on augmented training data ────────────────────────
X_tr_eval, X_te_eval, y_tr_eval, y_te_eval = train_test_split(
    X_train_aug, y_train_aug, test_size=0.2, random_state=1003
)

gp_eval = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval.fit(X_tr_eval, y_tr_eval)
mu_test, sigma_test = gp_eval.predict(X_te_eval, return_std=True)

rmse     = np.sqrt(mean_squared_error(y_te_eval, mu_test))
r2       = 1 - np.sum((y_te_eval - mu_test)**2) / np.sum((y_te_eval - y_te_eval.mean())**2)
mean_std = sigma_test.mean()

print(f"Test RMSE : {rmse:.4f}")
print(f"Test R²   : {r2:.4f}")
print(f"Mean σ    : {mean_std:.4f}")

# ── Full pool scores ───────────────────────────────────────────────────
ucb_norm_full = ucb_norm
d_norm_full   = d_norm

fig = plt.figure(figsize=(18, 11))
fig.suptitle("Iteration 2 — Surrogate Performance, Uncertainty & Acquisition Policy",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval, mu_test, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval.min(), mu_test.min()), max(y_te_eval.max(), mu_test.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse:.4f}  R²={r2:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test.min(), sigma_test.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals = y_te_eval - mu_test
ax_res.scatter(mu_test, residuals, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test.min(), mu_test.max(), 100),
    -2 * sigma_test.mean(), 2 * sigma_test.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper ───────────────────────────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 20 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6 ───────────────────────────────────────────────────────
dist_panel(ax_ucb, ucb_norm_full, ucb_norm_full[top20_pos_plot],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",      "#3498db")
dist_panel(ax_std, sigma,         sigma[top20_pos_plot],
           "σ(x)",                "Model Uncertainty\nσ distribution",        "#9b59b6")
dist_panel(ax_div, d_norm_full,   d_norm_full[top20_pos_plot],
           "$d_{norm}$(x)",       "Community Engagement\nDiversity distribution", "#2ecc71")
dist_panel(ax_s,   S,             S[top20_pos_plot],
           "S(x)",                "Combined Acquisition\nScore S(x)",         "#e67e22")

#plt.savefig("/mnt/user-data/outputs/iteration2_performance.png", dpi=150, bbox_inches="tight")
plt.show()
#print("Saved.")

In [ ]:
top20_local = np.argsort(S_prop)[::-1][:20]
top20_pos   = aligned_pos[top20_local]
top20_pool  = pool_idx_shuf[top20_pos]

results_iter2 = aligned_proposals.iloc[top20_local].copy().reset_index(drop=True)
results_iter2["y_pred"]        = mu_prop[top20_local]
results_iter2["y_uncert"]      = sigma_prop[top20_local]
results_iter2["ucb_norm"]      = ucb_norm_prop[top20_local]
results_iter2["d_norm"]        = d_norm_prop[top20_local]
results_iter2["cs_norm"]       = cs_norm_prop[top20_local]
results_iter2["coverage_norm"] = cov_norm_prop[top20_local]
results_iter2["score"]         = S_prop[top20_local]
top20_pos_plot                 = aligned_pos[top20_local]

In [ ]:
def make_explanation_iter2(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    n_scientists = len(row["scientist_id"].split(","))
    return (
        f"Selected via multi-objective acquisition (S={row['score']:.3f}) "
        f"balancing scientific discovery (UCB: mu={row['y_pred']:.3f}, sigma={row['y_uncert']:.3f}) "
        f"and community engagement (diversity={row['d_norm']:.3f}, "
        f"community_score={row['community_score']:.3f}, "
        f"proposed by {n_scientists} scientist(s): {row['scientist_id']}). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

# ── Build submission dataframe ─────────────────────────────────────────
submission_iter2 = results_iter2.copy()

# Rename feature columns to x1–x11
feat_rename = {features[i]: f"x{i+1}" for i in range(11)}
submission_iter2 = submission_iter2.rename(columns=feat_rename)

# Add model predictions aligned to top20
submission_iter2["y_pred"]   = mu_prop[top20_local]
submission_iter2["y_uncert"] = sigma_prop[top20_local]
submission_iter2["d_norm"]   = d_norm_prop[top20_local]
submission_iter2["sis_id"]   = "ansarir5"

submission_iter2["explanation"] = submission_iter2.apply(make_explanation_iter2, axis=1)

# ── Reorder to match submission spec ──────────────────────────────────
feat_cols = [f"x{i}" for i in range(1, 12)]
submission_iter2 = submission_iter2[
    ["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]
]

submission_iter2 = submission_iter2.sort_values("score", ascending=False).reset_index(drop=True)

print(submission_iter2[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

#submission_iter2.to_csv("/mnt/user-data/outputs/iteration2_submission.csv", index=False)
#print("\nSaved.")

In [ ]:
submission_iter2

In [ ]:
submission_iter2.to_csv("iteration2_submission.csv", index=False)
print("Saved.")

Before training the surrogate model, it is important to understand:

- The coverage of the RYB design space  
- The distribution of raw channel intensities  
- Correlations between channels  
- Whether the dataset contains clusters, gaps, or outliers  

Let's split the data before we get started so we don't have any leakage

## Iteration 3

Before we submit points for iteration 3, lets understand how happy the community was with the last set of points.

In [ ]:
# ── Parse happiness scores from text file ──────────────────────────────
happiness_raw2 = open("Proposals_iter2/happiness_results.txt").read()

happiness = {}
lines = happiness_raw2.strip().split("\n")
current_proposal = None
for line in lines:
    line = line.strip()
    if line.startswith("Proposal file:"):
        current_proposal = line.split("Proposal file:")[-1].strip().replace(".csv", "")
    elif line.startswith("Happiness with samples selected"):
        score = float(line.split(":")[-1].strip())
        happiness[current_proposal] = score

print("Parsed happiness scores:")
for k, v in sorted(happiness.items()):
    print(f"  {k}: {v:.4f}")

In [ ]:
#comparing iteration2_submission and happiness_results

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd


# ── Map happiness back to each selected point ──────────────────────────
# Points proposed by multiple scientists get the mean happiness
def mean_happiness(scientist_id_str):
    scientists = scientist_id_str.split(",")
    return np.mean([happiness[s.strip()] for s in scientists])

results_iter2["happiness"] = results_iter2["scientist_id"].apply(mean_happiness)

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Iteration 2 — Top 20 Selected Points vs Community Happiness",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_bar  = fig.add_subplot(gs[0, :2])   # happiness per selected point
ax_sci  = fig.add_subplot(gs[0, 2])    # mean happiness per scientist
ax_sco  = fig.add_subplot(gs[1, 0])    # happiness vs acquisition score
ax_ucb  = fig.add_subplot(gs[1, 1])    # happiness vs UCB
ax_div  = fig.add_subplot(gs[1, 2])    # happiness vs diversity

# ─── Panel 1: Happiness per selected point ────────────────────────────
colors = plt.cm.RdYlGn(
    (results_iter2["happiness"] - 1) / 4
)
bars = ax_bar.bar(
    range(len(results_iter2)),
    results_iter2["happiness"],
    color=colors, edgecolor="black", linewidth=0.6
)
ax_bar.axhline(results_iter2["happiness"].mean(), color="black",
               lw=1.5, ls="--",
               label=f"Mean happiness = {results_iter2['happiness'].mean():.2f}")
ax_bar.set_xticks(range(len(results_iter2)))
ax_bar.set_xticklabels(results_iter2["Pool_index"], rotation=45, ha="right", fontsize=8)
ax_bar.set_xlabel("Pool_index")
ax_bar.set_ylabel("Happiness score")
ax_bar.set_title("Happiness Score per Selected Point\n(colour: red=low, green=high)")
ax_bar.set_ylim(0, 5.5)
ax_bar.legend(fontsize=9)
ax_bar.grid(True, alpha=0.2, axis="y")

# ─── Panel 2: Mean happiness per scientist ────────────────────────────
sci_df = pd.DataFrame(happiness.items(), columns=["scientist_id", "happiness"])
sci_df = sci_df.sort_values("happiness", ascending=True)
bar_colors = plt.cm.RdYlGn((sci_df["happiness"] - 1) / 4)
ax_sci.barh(sci_df["scientist_id"], sci_df["happiness"],
            color=bar_colors, edgecolor="black", linewidth=0.6)
ax_sci.axvline(sci_df["happiness"].mean(), color="black", lw=1.5, ls="--",
               label=f"Mean = {sci_df['happiness'].mean():.2f}")
ax_sci.set_xlabel("Happiness score")
ax_sci.set_title("Happiness per Scientist")
ax_sci.set_xlim(0, 5.5)
ax_sci.legend(fontsize=9)
ax_sci.grid(True, alpha=0.2, axis="x")

# ─── Helper: scatter with correlation ─────────────────────────────────
def scatter_corr(ax, x, y, xlabel, title, color):
    ax.scatter(x, y, c=color, s=60, edgecolors="black", linewidths=0.6, alpha=0.8)
    # correlation
    r = np.corrcoef(x, y)[0, 1]
    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, "k--", lw=1.5, label=f"r = {r:.2f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Happiness")
    ax.set_title(title)
    ax.set_ylim(0, 5.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

# ─── Panels 3–5: Happiness vs score components ────────────────────────
scatter_corr(ax_sco, results_iter2["score"],    results_iter2["happiness"],
             "Acquisition score S(x)",  "Happiness vs Acquisition Score", "#e67e22")
scatter_corr(ax_ucb, results_iter2["ucb_norm"], results_iter2["happiness"],
             "UCB$_{norm}$(x)",         "Happiness vs Scientific Discovery", "#3498db")
scatter_corr(ax_div, results_iter2["d_norm"],   results_iter2["happiness"],
             "$d_{norm}$(x)",           "Happiness vs Diversity", "#2ecc71")

#plt.savefig("/mnt/user-data/outputs/iteration2_happiness.png", dpi=150, bbox_inches="tight")
plt.show()
#print("Saved.")

- We can see that mean happiness was 2.57 meaning overall the community was not super happy nor terribly displeased. 

- Overall proposals/scientists 7 and 8 were the happiest with the points and proposals/scientists 6, 5, 10 and 12 were fairly satisfied. 

- This is important to note as the last iteration placed more emphasis on happiness over discovery but was not able to achieve it. 

- Instead of trying to force happiness, lets try to understand why they aren't very happy and adjust the acquisition policy from there. 

- Important to note that happiness was found to be inversely proportional to scientific discovery but thats probably because the policy didnt emphasize scientific discovery enough. 

- Diversity is the strength in the policy so dnorm will be kept the same. 